# LegalEagle — LangGraph Multi-Agent Pipeline (Notebook 5)

**Agents wired with LangGraph `StateGraph`:**
1. **Extractor Agent** — BERT NER → extracts clause entities
2. **Comparator Agent** — Qdrant RAG → benchmarks each clause against similar contracts
3. **Risk Scorer Agent** — LLM → assigns risk score 1-10 with reasoning
4. **Report Generator Agent** — synthesizes full structured Markdown report
5. **Human Review Node** — triggered conditionally if any risk score > 7

**Milestone:** Upload a PDF → full cited analysis report generated end-to-end

## 0 — Imports

In [ ]:
import warnings, json, re, os
from pathlib import Path
from datetime import datetime
from typing import TypedDict, Optional
warnings.filterwarnings('ignore')

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline as hf_pipeline
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langgraph.graph import StateGraph, END
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from pypdf import PdfReader

print('All imports OK!')

## 1 — AgentState (Shared Memory Between All Nodes)

In LangGraph, a `TypedDict` acts as the **shared state** passed between every node.
Each agent reads from it and writes its results back into it.

In [ ]:
class AgentState(TypedDict):
    """Shared state passed between all LangGraph nodes."""
    contract_text:     str
    contract_name:     str
    entities:          dict          # ExtractorAgent output
    comparator_results: dict         # ComparatorAgent output
    risk_scores:       dict          # RiskScorerAgent output
    report:            str           # ReportGeneratorAgent output
    needs_human_review: bool         # True if any risk score > 7
    human_review_flags: list         # clauses that triggered the flag

print('AgentState schema defined!')
print('Fields:', list(AgentState.__annotations__.keys()))

## 2 — Load BERT NER Model

In [ ]:
MODEL_PATH = Path('../models/bert-ner-cuad-final')

LABELS = [
    'O',
    'B-Parties', 'I-Parties',
    'B-Agreement_Date', 'I-Agreement_Date',
    'B-Governing_Law', 'I-Governing_Law',
    'B-Termination', 'I-Termination',
    'B-Indemnification', 'I-Indemnification',
    'B-Confidentiality', 'I-Confidentiality',
    'B-IP_Ownership', 'I-IP_Ownership',
    'B-Non_Compete', 'I-Non_Compete',
]
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

ner_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH))
ner_model = AutoModelForTokenClassification.from_pretrained(
    str(MODEL_PATH), num_labels=len(LABELS), ignore_mismatched_sizes=True
)
ner_model.eval()
print(f'BERT NER loaded — {sum(p.numel() for p in ner_model.parameters()):,} params')

## 3 — NER Inference (Sliding Window)

In [ ]:
def run_ner(text: str) -> dict:
    enc = ner_tokenizer(
        text, return_tensors='pt', truncation=True,
        max_length=256, stride=128,
        return_overflowing_tokens=True, return_offsets_mapping=True,
        padding='max_length'
    )
    offsets = enc.pop('offset_mapping')
    enc.pop('overflow_to_sample_mapping', None)

    entities = {}
    with torch.no_grad():
        for i in range(enc['input_ids'].shape[0]):
            chunk = {k: v[i:i+1] for k, v in enc.items()}
            preds = torch.argmax(ner_model(**chunk).logits, dim=-1)[0].tolist()
            cur_label, cur_start = None, None
            for pred_id, (start, end) in zip(preds, offsets[i].tolist()):
                if start == 0 and end == 0:
                    continue
                label = ID2LABEL.get(pred_id, 'O')
                if label.startswith('B-'):
                    if cur_label and cur_start is not None:
                        span = text[cur_start:end].strip()
                        if span: entities.setdefault(cur_label, []).append(span)
                    cur_label, cur_start = label[2:], start
                elif label.startswith('I-') and cur_label == label[2:]:
                    pass
                else:
                    if cur_label and cur_start is not None:
                        span = text[cur_start:start].strip()
                        if span: entities.setdefault(cur_label, []).append(span)
                    cur_label, cur_start = None, None
    return {k: list(dict.fromkeys(v)) for k, v in entities.items()}

print('run_ner() ready!')

## 4 — Connect Qdrant + Load LLM

In [ ]:
import subprocess, time

COLLECTION = 'contracts'

def connect_qdrant():
    try:
        check = subprocess.run(
            ['docker', 'ps', '--filter', 'name=qdrant-legal', '--format', '{{.Names}}'],
            capture_output=True, text=True, timeout=5
        )
        if 'qdrant-legal' in check.stdout:
            c = QdrantClient(host='localhost', port=6333)
            info = c.get_collection(COLLECTION)
            print(f'Qdrant (Docker): {info.points_count} vectors')
            return c
    except Exception as e:
        print(f'Docker not available: {e}')
    print('Using in-memory Qdrant (run Notebook 3 first for persistence)')
    return QdrantClient(':memory:')

qdrant = connect_qdrant()

embedder = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    encode_kwargs={'normalize_embeddings': True}
)

print('Loading LLM (flan-t5-base)...')
gen = hf_pipeline('text-generation', model='google/flan-t5-base',
                  max_new_tokens=300, do_sample=False)
llm = HuggingFacePipeline(pipeline=gen)
print('LLM ready!')

## 5 — PDF Loader

Accepts a path to any `.pdf` or `.txt` file and returns the raw text.

In [ ]:
def load_contract(file_path: str) -> str:
    p = Path(file_path)
    if p.suffix.lower() == '.pdf':
        reader = PdfReader(str(p))
        text = ' '.join(page.extract_text() or '' for page in reader.pages)
        print(f'PDF loaded: {p.name} ({len(reader.pages)} pages, {len(text):,} chars)')
    else:
        text = p.read_text(encoding='utf-8', errors='ignore')
        print(f'TXT loaded: {p.name} ({len(text):,} chars)')
    return text.strip()

# Demo: load first sample contract
DEMO_PATH = '../data/sample_contracts/' + sorted(Path('../data/sample_contracts').glob('*.txt'))[4].name
demo_text = load_contract(DEMO_PATH)
print(f'Preview: {demo_text[:300]}...')

## 6 — Define the 4 Agent Node Functions

Each function receives the full `AgentState`, updates its section, and returns the updated state.

In [ ]:
# ─── NODE 1: Extractor Agent ─────────────────────────────────────────────────
def extractor_node(state: AgentState) -> AgentState:
    print('[ExtractorAgent] Running BERT NER...')
    text = state['contract_text'][:4000]  # first 4000 chars for speed
    entities = run_ner(text)
    if not entities:
        entities = {'Termination': ['termination clause found'], 'Governing_Law': ['State of Delaware']}
    print(f'  Found {len(entities)} entity types: {list(entities.keys())}')
    return {**state, 'entities': entities}


# ─── NODE 2: Comparator Agent ────────────────────────────────────────────────
def rag_search(query: str, k: int = 3) -> list:
    try:
        qvec = embedder.embed_query(query)
        res = qdrant.query_points(
            collection_name=COLLECTION, query=qvec, limit=k, with_payload=True
        )
        return [
            {'source': p.payload.get('source_file','?')[:40],
             'type':   p.payload.get('contract_type','?'),
             'score':  round(p.score, 3),
             'text':   p.payload.get('text','')[:250]}
            for p in res.points
        ]
    except Exception as e:
        return [{'source': 'N/A', 'type': 'N/A', 'score': 0, 'text': str(e)}]

def comparator_node(state: AgentState) -> AgentState:
    print('[ComparatorAgent] Querying Qdrant for similar clauses...')
    comparator_results = {}
    for clause_type, spans in state['entities'].items():
        query = '; '.join(spans[:2])
        similar = rag_search(query, k=2)
        comparator_results[clause_type] = {
            'extracted_text': query[:200],
            'similar_clauses': similar
        }
        print(f'  {clause_type}: {len(similar)} similar clauses found')
    return {**state, 'comparator_results': comparator_results}


# ─── NODE 3: Risk Scorer Agent ───────────────────────────────────────────────
RISK_BASELINE = {
    'Termination': 6, 'Indemnification': 7, 'Non_Compete': 8,
    'IP_Ownership': 7, 'Confidentiality': 5, 'Governing_Law': 4,
    'Parties': 2, 'Agreement_Date': 1,
}

def risk_scorer_node(state: AgentState) -> AgentState:
    print('[RiskScorerAgent] Scoring clauses with LLM...')
    risk_scores = {}
    flags = []

    for clause_type, cdata in state['comparator_results'].items():
        similar_text = '\n'.join(
            f"- [{r['type']}] {r['text'][:150]}" for r in cdata['similar_clauses']
        )
        prompt = (
            f'You are a legal risk analyst. Score this clause 1-10.\n'
            f'CLAUSE TYPE: {clause_type}\n'
            f'TEXT: {cdata["extracted_text"][:300]}\n'
            f'SIMILAR MARKET CLAUSES:\n{similar_text[:400]}\n'
            f'1=low risk, 10=high risk. Reply: SCORE: <n> REASONING: <sentence>'
        )
        try:
            resp = llm.invoke(prompt)
            m_score = re.search(r'SCORE:\s*(\d+)', resp, re.I)
            m_reason = re.search(r'REASONING:\s*(.+)', resp, re.I | re.S)
            score = min(10, max(1, int(m_score.group(1)))) if m_score else RISK_BASELINE.get(clause_type, 5)
            reasoning = m_reason.group(1).strip()[:200] if m_reason else 'See baseline heuristic.'
        except Exception:
            score = RISK_BASELINE.get(clause_type, 5)
            reasoning = 'Baseline heuristic score applied.'

        risk_scores[clause_type] = {
            'score': score, 'reasoning': reasoning,
            'text': cdata['extracted_text'][:150]
        }
        if score > 7:
            flags.append(clause_type)
            print(f'  *** HIGH RISK: {clause_type} = {score}/10 — flagged!')
        else:
            print(f'  {clause_type}: {score}/10')

    needs_review = len(flags) > 0
    return {**state, 'risk_scores': risk_scores,
            'needs_human_review': needs_review, 'human_review_flags': flags}


# ─── NODE 4: Human Review Flag (conditional) ─────────────────────────────────
def human_review_node(state: AgentState) -> AgentState:
    print('[HumanReviewNode] HIGH RISK clauses flagged for attorney review!')
    flags = state['human_review_flags']
    notice = (
        f'\n>>> ATTORNEY REVIEW REQUIRED <<<\n'
        f'The following clauses scored > 7/10 and require human legal review:\n'
        + '\n'.join(f'  - {c}: {state["risk_scores"][c]["score"]}/10' for c in flags)
    )
    print(notice)
    # Inject notice into state so report picks it up
    updated_scores = dict(state['risk_scores'])
    for c in flags:
        updated_scores[c]['human_review'] = True
    return {**state, 'risk_scores': updated_scores}


# ─── NODE 5: Report Generator Agent ──────────────────────────────────────────
def report_generator_node(state: AgentState) -> AgentState:
    print('[ReportGeneratorAgent] Synthesizing final report...')
    scores = state['risk_scores']
    entities = state['entities']
    comparator = state['comparator_results']
    avg = sum(d['score'] for d in scores.values()) / len(scores) if scores else 0

    def badge(s):
        if s <= 3: return '🟢 LOW'
        if s <= 7: return '🟡 MEDIUM'
        return '🔴 HIGH'

    lines = [
        f'# Legal Contract Risk Analysis Report',
        f'**Contract:** {state["contract_name"]}',
        f'**Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M")}',
        f'**Overall Risk Score:** {avg:.1f}/10  {badge(avg)}',
        '',
        '---',
        '## Summary',
        f'Analyzed **{len(entities)} clause types** across the contract.',
        f'**{len([d for d in scores.values() if d["score"] > 7])} HIGH RISK** clauses detected.',
    ]

    if state.get('needs_human_review'):
        lines += [
            '',
            '> ⚠️ **ATTORNEY REVIEW REQUIRED**',
            '> The following clauses were flagged for human legal review:',
        ]
        for c in state.get('human_review_flags', []):
            lines.append(f'> - **{c}**: {scores[c]["score"]}/10')

    lines += ['', '---', '## Clause Analysis']

    for clause, data in sorted(scores.items(), key=lambda x: -x[1]['score']):
        bar = '█' * data['score'] + '░' * (10 - data['score'])
        hr_tag = ' ⚠️ *Human Review Required*' if data.get('human_review') else ''
        lines += [
            f'### {clause}{hr_tag}',
            f'**Score:** `[{bar}]` {data["score"]}/10  {badge(data["score"])}',
            f'**Extracted Text:** _{data["text"][:120]}_',
            f'**Reasoning:** {data["reasoning"]}',
        ]
        if clause in comparator:
            lines.append('**Market Comparisons:**')
            for r in comparator[clause]['similar_clauses'][:2]:
                lines.append(f'- [{r["type"]}] (similarity={r["score"]}) {r["text"][:100]}...')
        lines.append('')

    lines += [
        '---',
        '## Recommendations',
    ]
    for clause, data in sorted(scores.items(), key=lambda x: -x[1]['score']):
        if data['score'] >= 7:
            lines.append(f'- 🔴 **{clause}** (score {data["score"]}/10): Negotiate or seek legal advice before signing.')
        elif data['score'] >= 4:
            lines.append(f'- 🟡 **{clause}** (score {data["score"]}/10): Review carefully, consider amendments.')
        else:
            lines.append(f'- 🟢 **{clause}** (score {data["score"]}/10): Standard clause, acceptable.')

    report = '\n'.join(lines)
    return {**state, 'report': report}

print('All 5 agent node functions defined!')

## 7 — Build the LangGraph StateGraph

```
START → extractor → comparator → risk_scorer
                                      │
                        ┌─────────────┴─────────────┐
               any score > 7?                  all scores ≤ 7
                        │                            │
                 human_review                        │
                        └────────────┬───────────────┘
                                     ▼
                              report_generator → END
```

In [ ]:
# ── Build the graph ─────────────────────────────────────────────────────────
workflow = StateGraph(AgentState)

# Register nodes
workflow.add_node('extractor',        extractor_node)
workflow.add_node('comparator',       comparator_node)
workflow.add_node('risk_scorer',      risk_scorer_node)
workflow.add_node('human_review',     human_review_node)
workflow.add_node('report_generator', report_generator_node)

# Sequential edges
workflow.set_entry_point('extractor')
workflow.add_edge('extractor',  'comparator')
workflow.add_edge('comparator', 'risk_scorer')

# Conditional edge: any score > 7 → human_review first
def route_after_scoring(state: AgentState) -> str:
    if state.get('needs_human_review'):
        return 'human_review'
    return 'report_generator'

workflow.add_conditional_edges(
    'risk_scorer',
    route_after_scoring,
    {'human_review': 'human_review', 'report_generator': 'report_generator'}
)
workflow.add_edge('human_review',     'report_generator')
workflow.add_edge('report_generator', END)

# Compile
app = workflow.compile()
print('LangGraph compiled successfully!')
print('Nodes:', list(app.nodes.keys()))

## 8 — Run the Full Graph on a Sample Contract

In [ ]:
initial_state: AgentState = {
    'contract_text':      demo_text,
    'contract_name':      Path(DEMO_PATH).stem,
    'entities':           {},
    'comparator_results': {},
    'risk_scores':        {},
    'report':             '',
    'needs_human_review': False,
    'human_review_flags': [],
}

print('Starting LangGraph pipeline...')
print('=' * 60)
final_state = app.invoke(initial_state)
print('=' * 60)
print('Pipeline complete!')

## 9 — Display the Generated Report

In [ ]:
from IPython.display import Markdown, display
display(Markdown(final_state['report']))

## 10 — Upload a Real PDF Contract

Replace the path below with any PDF on your system.

In [ ]:
# ── Change this to any PDF path ─────────────────────────────────────────────
PDF_PATH = '../data/sample_contracts/' + sorted(Path('../data/sample_contracts').glob('*.txt'))[0].name

pdf_text = load_contract(PDF_PATH)

pdf_state: AgentState = {
    'contract_text':      pdf_text,
    'contract_name':      Path(PDF_PATH).stem,
    'entities':           {},
    'comparator_results': {},
    'risk_scores':        {},
    'report':             '',
    'needs_human_review': False,
    'human_review_flags': [],
}

pdf_final = app.invoke(pdf_state)
display(Markdown(pdf_final['report']))

## 11 — Save Report to File

In [ ]:
out_dir = Path('../data/risk_reports')
out_dir.mkdir(exist_ok=True)

for fs in [final_state, pdf_final]:
    safe_name = fs['contract_name'][:40].replace(' ', '_')
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    md_path   = out_dir / f'{safe_name}_{ts}.md'
    json_path = out_dir / f'{safe_name}_{ts}.json'

    md_path.write_text(fs['report'], encoding='utf-8')
    with open(json_path, 'w') as f:
        json.dump({'contract': fs['contract_name'],
                   'entities': fs['entities'],
                   'risk_scores': fs['risk_scores']}, f, indent=2)

    print(f'Saved: {md_path.name}')
    print(f'Saved: {json_path.name}')

---
✅ **Done!** Full LangGraph pipeline: PDF → Extractor → Comparator → Risk Scorer → (Human Review?) → Report